In [28]:
import torch
import torch.nn as nn

from training import WaveformDataset, FeatureDataset, train_epoch, evaluate_model, train_model
from models import VGG, HubertAudioClassifier, ASTAudioClassifier, vgg_make_layers, HubertAudioClassifierWithAudioType
from data_preprocessing import process_data, partition_by_source
from config import SEED
from utils import set_seeds

set_seeds(SEED)

# VGG analysis

We proceed by writing a sample of code using our pipelines to train and validate different VGG models. First, we specify the paramters for the features we will be using, and load our dataframes with the data preprocessing pipeline.

In [ ]:
class_to_id = {'pos': 0, 'neg': 1}
mfcc_params = {
    "n_mfcc": 13,
    "lifter": 22,
    "preemph": 0.97,
    "n_fft": 2048,
    "n_mels": 80,
    "win_size": 0.025,
    "win_stride": 0.01,
    "use_cached": True,
}

feats_dir = 'data/features/'
train_df, test_df = process_data(mfcc_params, class_to_id, feats_dir, feature_type = "mfsc", target_sr = 16000, dtw_computed = True)

We then define the arguments for training.

In [35]:
args = {
    "feature_col": "mfsc",
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "max_len": None,
    "percentile": 90,
    "batch_size": 32,
    "optimizer": "adam",
    "lr": 1e-4,
    "momentum": 0.9,
    "epochs": 100,
    "patience": 5,
    "log_interval": 5,
    "verbose": True,
}

Finally, we define the arguments of our VGG model. We will experiment with different configurations, playing with depth, local and global pooling, dropout, size of the classifier, etc.

In [36]:
seed = 304
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

model = VGG(
    vgg_name="VGG13",
    hidden=64,
    pool_type="max",
    end_pooling="avg",
    dropout_after_pool=False,
    dropout_p=0.2,
    weight_initialization=False,
)

In [ ]:
criterion = nn.BCEWithLogitsLoss(reduction='mean')
model, state = train_model(train_df, test_df, model, criterion, **args)

## Experiments with VGG models

We compare the performance of four models: VGG11, VGG13, VGG16 and VGG19. All models use *Max Pooling* inside the feature extractor, and a concatenation of *Global Average Pooling (GAP)* and *Global Max Pooling (GMP)* before the classifier (hidden size = 64).

Models are trained for full epochs and evaluated on a validation set after each epoch. Training proceeds with **early stopping (patience = 5)** based on validation AUC, and the best model checkpoint is restored. Optimization uses **Adam** with a learning rate of **1e-4**.

---

**Weight Initialization.**

First, we test a custom He initialization for all four VGG models, but we find it leads to slightly lower validation AUC than PyTorch’s default initialization. This suggests that the default initialization is already well suited for this setup. The best performing model with our default paramters is **VGG13**.

| Model | Weight Init | Valid AUC | Valid Loss | Epoch | Time (s) |
|--------|--------------|-----------|-------------|--------|-----------|
| **VGG11** | ✓ Yes | 69.01% | 0.66254 | 5 | 63.9 |
|           | ✗ No  | 70.26% | 0.66758 | 10 | 128.2 |
| **VGG13** | ✓ Yes | 70.84% | 0.73743 | 14 | 299.91 |
|           | ✗ No  | **72.12%** | 0.64245 | 6 | 127.94 |
| **VGG16** | ✓ Yes | 68.74% | 0.65993 | 5 | 125.37 |
|           | ✗ No  | 71.05% | 0.69823 | 15 | 379.55 |
| **VGG19** | ✓ Yes | 70.37% | 0.63888 | 13 | 378.07 |
|           | ✗ No  | 69.14% | 0.69127 | 8 | 239.92 |

---

**Local Pooling Type**

We proceed without our custom weight initialization. Now, we see the difference of having Max Pooling and Average Pooling within the convolutional blocks of the feature extractor, finding minimal AUC differences. All the comparisons are done with GAP+GMP concatenation before the classifier. This suggests that the deeper feature extractor compensates for pooling type. 

| Model  | Local Pooling | Valid AUC | Valid Loss | Epoch | Time (s) |
|---------|------------|---------------|-------------|--------|-----------|
| **VGG11** | max | 70.26% | 0.6676 | 10 | 128.2 |
|  | avg | **71.34%** | 0.6499 | 17 | 217.4 |
| **VGG13** | max | **72.12%** | 0.6425 | 6 | 127.9 |
|  | avg | 71.58% | 0.6840 | 15 | 320.7 |
| **VGG16** | max | 71.05% | 0.6982 | 15 | 379.6 |
|  | avg | **71.39%** | 0.6468 | 17 | 429.5 |
| **VGG19** | max | 69.14% | 0.6913 | 8 | 232.9 |
|  | avg | **70.71%** | 0.6395 | 15 | 437.0 |

---

**Global Pooling Configurations**

We perform a full sweep of local/global pooling combinations for VGG11. We observee that using **local max pooling** and **global average pooling** at the output provided the best AUC, **73.56%**. This likely reflects that local discriminative cues (cough bursts, formant peaks) benefit from max pooling, while averaging at the global scale prevents overfitting to transient spikes. Additionally, using one single global pooling rather than the concatenation reduces the size of the classifier, which may help with overfitting. We also test GAP+GMP concatenation reducing the hidden size of the classifier to 32 to see if we achieve a similar AUC, and we do for **local average pooling**, obtaining **73.43%**.

| Model | Local Pooling | Global Pooling | Hidden | Valid AUC | Valid Loss | Epoch | Time (s) |
|--------|---------------|----------------|---------|----------------|-------------|--------|-----------|
| **VGG11** | max | both | 64 | 70.26% | 0.6676 | 10 | 128.2 |
|           | avg | both | 64 | 71.34% | 0.6499 | 17 | 217.4 |
|           | max | max | 64 | 70.26% | 0.6410 | 19 | 243.6 |
|           | avg | max | 64 | 69.32% | 0.6669 | 10 | 127.3 |
|           | max | avg | 64 | **73.56%** | 0.6530 | 17 | 217.7 |
|           | avg | avg | 64 | 72.19% | 0.6591 | 6 | 76.5 |
|           | max | both | 32 | 71.86% | 0.6559 | 12 | 153.9 |
|           | avg | both | 32 | **73.43%** | 0.6424 | 26 | 333.94 |


---

**Dropout in Feature Extractor**

We train all four VGG models with **local max pooling** and **global average pooling**, and compare the results with and without adding droput (p=0.2) after pooling layers in the feature extractor (the classifier includes has dropout). We used a hidden size of 64 for the classifier. The best validation AUC (**74.83%**) was achieved by VGG13. Dropout didn't seem to make a big difference for the other three models. We also tried VGG13 with local average pooling with GAP+GMP concatenation and a hidden size of 32, but got **71.19%** AUC after 15 epochs.

| Model  | Dropout after Pool | Valid AUC | Valid Loss | Epoch | Time (s) |
|---------|--------------------|---------------|-------------|--------|-----------|
| **VGG11** | No | 73.56% | 0.6529 | 17 | 217.7 |
|  | 0.2 | **73.58%** | 0.6305 | 30 | 393.8 |
| **VGG13** | No | **74.83%** | 0.6061 | 24 | 514.7 |
|  | 0.2 | 72.57% | 0.6320 | 16 | 345.5 |
| **VGG16** |  No | 72.49% | 0.6673 | 12 | 302.1 |
|  | 0.2 | **72.87%** | 0.6298 | 19 | 486.0 |
| **VGG19** | No | 69.04% | 0.6493 | 7 | 202.7 |
|  | 0.2 | **69.11%** | 0.6486 | 7 | 204.1 |

## Feature analysis

Our best performing model so far is VGG13 with local max pooling and global average pooling. We now want to see the effect that the features have on the accuracy of the model. Until this point, we have been using MFSC (log-Mel energies) with 80 Mel bins (`n_mels = 80`)

We compare MFSC (log-Mel energies), MFCC, and raw Mel spectrogram features for our best performing model. MFSC with 80 Mel bins achieved a baseline validation AUC of **74.8%**, outperforming MFCC (**69.9%**, we use `n_mfcc` to 80 to make an accurate comparison) and raw Mel spectrogram (**66.3%**). The key difference is the logarithmic compression in MFSC, which stabilizes training and emphasizes subtle spectral cues critical for CNNs. Increasing the number of Mel bins (`n_mels`) to 120 slightly improved validation AUC to **75.2%** after 28 epochs (increassing `patience` to 10), likely due to higher spectral resolution, which is helpful for convolutional architectures. 

These results align with common practices in speech recognition, where MFSC is preferred for convolutional architectures, while MFCCs are typically used for tasks relying on cepstral analysis.

In [ ]:
class_to_id = {'pos': 0, 'neg': 1}
mfcc_params = {"n_mfcc": 13, "lifter": 22, "preemph": 0.97, "n_fft": 2048,
    "n_mels": 120, "win_size": 0.025, "win_stride": 0.01, "use_cached": True}

feats_dir = 'data/features/'
train_df, test_df = process_data(mfcc_params, class_to_id, feats_dir, feature_type = "mfsc", target_sr = 16000, dtw_computed = True)

args = {"feature_col": "mfsc", "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "max_len": None, "percentile": 90, "batch_size": 32, "optimizer": "adam", "lr": 1e-4, "momentum": 0.9,
    "epochs": 100, "patience": 10, "log_interval": 5, "verbose": True}

seed = 304
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

model = VGG(vgg_name="VGG13", hidden=64, pool_type="max", end_pooling="avg", dropout_after_pool=False, dropout_p=0.2, weight_initialization=False)

In [ ]:
criterion = nn.BCEWithLogitsLoss(reduction='mean')
model, state = train_model(train_df, test_df, model, criterion, **args)

# HuBERT Analysis

For this set of tests, we finetune a pretrained Hubert model, using it as a feature extractor for our task of audio classification.


In [ ]:
model = HubertAudioClassifier(adapter_hidden_size=64)
model.freeze_feature_encoder()                   # Freeze the feature extractor
for param in model.hubert.encoder.parameters():  # Freeze the encoder
    param.requires_grad = False

In [55]:
model

HubertAudioClassifier(
  (hubert): HubertModel(
    (feature_extractor): HubertFeatureEncoder(
      (conv_layers): ModuleList(
        (0): HubertGroupNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (activation): GELUActivation()
          (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
        )
        (1-4): 4 x HubertNoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
        (5-6): 2 x HubertNoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): HubertFeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=512, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): Huber

We process our data into train and test dataframes.

In [ ]:
class_to_id = {'pos': 0, 'neg': 1}
mfcc_params = {
    "n_mfcc": 13,
    "lifter": 22,
    "preemph": 0.97,
    "n_fft": 2048,
    "n_mels": 80,
    "win_size": 0.025,
    "win_stride": 0.01,
    "use_cached": True,
}

feats_dir = 'data/features/'
train_df, test_df = process_data(mfcc_params, class_to_id, feats_dir, feature_type = "raw_waveform", target_sr = 16000, dtw_computed = True)

We define the arguments for training.

In [57]:
args = {
    "feature_col": "raw_waveform",
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "max_len": None,
    "percentile": 90,
    "batch_size": 8,
    "optimizer": "adam",
    "lr": 1e-4,
    "momentum": 0.9,
    "epochs": 100,
    "patience": 10,
    "log_interval": 5,
    "verbose": True,
    "processor_name": "facebook/hubert-base-ls960",
}

In [ ]:
criterion = nn.BCEWithLogitsLoss(reduction='mean')
model, state = train_model(train_df, test_df, model, criterion, **args)

## Experiments with HubertAudioClassifier
We finetune a **pretrained Hubert model**, using it as a feature extractor. We freeze the encoder and feature extractor, and train an adapter layer and classifier using batches of size 8. To maximize the validation AUC, we: 
* Add a **2-layer adapter** with residual connection to fine-tune the model.
* Apply **Layer Normalization** after the residual connection to stabilize training.
* Use **GELU activations** for smoother non-linearity.
* Apply **global average pooling (GAP) + global max pooling (GMP)** concatenation over the temporal dimension to summarize sequence features.
* Use **dropout to 0.2** for better regularization.

This fine-tuned HubertAudioClassifier achieved **75.88% AUC on the validation set** (95% confidence interval: 71.41% - 80.36%), surpassing the best VGG-based baseline (around 75.2%) with more consistent performance across runs. Compared to VGGs, the self-supervised Hubert features capture temporal structure more effectively, which is good for **multisecond** audios (like our case) and likely explains the improved stability and AUC.

# AST Analysis

In [ ]:
model = ASTAudioClassifier(adapter_hidden_size=64)
for param in model.ast.parameters():  # Freeze the feature extractor
    param.requires_grad = False

In [3]:
model

ASTAudioClassifier(
  (ast): ASTModel(
    (embeddings): ASTEmbeddings(
      (patch_embeddings): ASTPatchEmbeddings(
        (projection): Conv2d(1, 768, kernel_size=(16, 16), stride=(10, 10))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ASTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ASTLayer(
          (attention): ASTAttention(
            (attention): ASTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (output): ASTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ASTIntermediate(
            (dense): Linear(in_features=768, out

We process our data into train and test dataframes.

In [ ]:
class_to_id = {'pos': 0, 'neg': 1}
mfcc_params = {
    "n_mfcc": 13,
    "lifter": 22,
    "preemph": 0.97,
    "n_fft": 2048,
    "n_mels": 80,
    "win_size": 0.025,
    "win_stride": 0.01,
    "use_cached": True,
}

feats_dir = 'data/features/'
train_df, test_df = process_data(mfcc_params, class_to_id, feats_dir, feature_type = "raw_waveform", target_sr = 16000, dtw_computed = True)

We define the arguments for training.

In [18]:
args = {
    "feature_col": "raw_waveform",
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "max_len": None,
    "percentile": 90,
    "batch_size": 32,
    "optimizer": "adam",
    "lr": 1e-4,
    "momentum": 0.9,
    "epochs": 100,
    "patience": 10,
    "log_interval": 5,
    "verbose": True,
    "processor_name": "MIT/ast-finetuned-audioset-10-10-0.4593",
}

In [19]:
print(args["processor_name"])

MIT/ast-finetuned-audioset-10-10-0.4593


In [45]:
import importlib
import models
import data_preprocessing

importlib.reload(models)
importlib.reload(training)
importlib.reload(data_preprocessing)

<module 'data_preprocessing' from '/u/mtomasbernal/nlp/data_preprocessing.py'>

In [ ]:
criterion = nn.BCEWithLogitsLoss(reduction='mean')
model, state = train_model(train_df, test_df, model, criterion, **args)

## Experiments with ASTAudioClassifier
We finetune a **pretrained AST model**, using it as a feature extractor for audio classification. We train the model with an adapter layer and classifier while keeping the AST encoder frozen, using batches of size 32. To maximize the validation AUC, we: 
* Add a **2-layer adapter** with residual connection to fine-tune the model.
* Apply **Layer Normalization** after the residual connection to stabilize training.
* Use **GELU activations** for smoother non-linearity.
* The AST model uses its own pooling mechanism, which is designed to summarize sequence features over time.
* Use **dropout to 0.2** for better regularization.

This fine-tuned ASTAudioClassifier achieved **70.88% AUC on the validation set** (95% confidence interval: 66.06% - 75.69%). Notably, the model converged after just **two epochs** of training. Despite the fast convergence, AST's performance plateaus at around 71%, achieving no improvement after 10 more epochs, even with `ReduceLROnPlateau`. In contrast, both the best VGG model and HubertAudioClassifier obtained higher validation AUC, though they took longer to train. It is likely that the GMP + GAP concatenation pooling used in HuBERT may have contributed to its higher final AUC, and the comparison between global pooling strategies for VGG models seems to confirm that.

## Cross-domain generalization: Source analysis

We now proceed to analyze the capacity of our models to generalize to different domains. For that, in the preprocessing process we added a `source_id` column in the dataframes. We use the following function to create a data partition that allows us to test that.

In [ ]:
train_df, test_df = partition_by_source(train_df, test_df, 1)

This will allow us to train on the audios from all sources except the selected source (in this case, 1). There are 4 source ids, and ids 1 and 2 refer to the same source, so we will group the audios with `source_id` 1 and 2.

In [ ]:
args = {
    "feature_col": "raw_waveform",
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "max_len": None,
    "percentile": 90,
    "batch_size": 32,
    "optimizer": "adam",
    "lr": 1e-4,
    "momentum": 0.9,
    "epochs": 100,
    "patience": 7,
    "log_interval": 5,
    "verbose": True,
    "processor_name": "MIT/ast-finetuned-audioset-10-10-0.4593",
}

criterion = nn.BCEWithLogitsLoss(reduction='mean')
model, state = train_model(train_df, test_df, model, criterion, **args)

We train the `ASTAudioClassifier` and `HubertAudioClassifier` on these data partitions, so the models don't see any audio from sources 1 & 2, 3 and 4 during training, and then evaluate the models on the audios from the unseen sources. We show the results in the following table:

| Model  | Target source | Test audios | Train audios | Epoch | AUC |
|---------|--------------------|---------------|-------------|--------|-----------|
| **AST** | 1 & 2 | 464 | 2005 | 1 | 42.56% |
|         | 3 | 1078 | 1391 | 1 | 48.12% |
|         | 4 | 927 | 1542 | 3 | 46.41% |
| **HuBERT** | 1 & 2 | 464 | 2005 | 4 | 44.42% |
|            | 3 | 1078 | 1391 | 8 | 53.60% |
|            | 4 | 927 | 1542 | 13 | 48.87% |

The generalization performance becomes considerably worse. When training on data from all sources, we obtained a validation AUC of ~75% for HuBERT and ~71% for AST. This shows the difficulties of cross-domain generalization. This is likely due to the variations in the recordings from the different sources: original formats, microphones used to record, ambient noise, etc. There are multiple ways we could adress this problem:

1) Train on **more data and from more different sources**. We are working with ~2000 audio files from 3 different sources, which is not enough to obtain domain invariant features.

2) **Fine-tune the model** on audios from the source we are interested in. If we want to classify correctly audios from a given source, we saw that training the model on those audios allowed us to obtain around ~75% validation AUC. Before using the model to classify audios from a given source, just make sure to train it on some samples from that source beforehand.

3) Perform **Data augmentation** to simulate different conditions (and different "sources"). We can add noise, change the pitch or the reverberation to simulate recordings with different microphones, in different rooms, etc.

4) Train a **Domain Adversarial Neural Network (DANN)** to learn domain invariant features.

## Does `audio_type` help?

In the preprocessing, we added an `audio_type` column to our dataframe, that labels each audio as either a "cough" or a "breath". We add this information as an embedding to a custom model, `HubertAudioClassifierWithAudioType`, to see if this information is helpful. We embed the audio type before the adapter layer, which allows the model to learn custom features for each audio type.

In [ ]:
model = HubertAudioClassifierWithAudioType(adapter_hidden_size=64)
model.freeze_feature_encoder()                   # Freeze the feature extractor
for param in model.hubert.encoder.parameters():  # Freeze the encoder
    param.requires_grad = False

In [61]:
model

HubertAudioClassifierWithAudioType(
  (hubert): HubertModel(
    (feature_extractor): HubertFeatureEncoder(
      (conv_layers): ModuleList(
        (0): HubertGroupNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (activation): GELUActivation()
          (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
        )
        (1-4): 4 x HubertNoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
        (5-6): 2 x HubertNoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): HubertFeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=512, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (en

In [ ]:
class_to_id = {'pos': 0, 'neg': 1}
mfcc_params = {
    "n_mfcc": 13,
    "lifter": 22,
    "preemph": 0.97,
    "n_fft": 2048,
    "n_mels": 80,
    "win_size": 0.025,
    "win_stride": 0.01,
    "use_cached": True,
}

feats_dir = 'data/features/'
train_df, test_df = process_data(mfcc_params, class_to_id, feats_dir, feature_type = "raw_waveform", target_sr = 16000, dtw_computed = True)

In [62]:
args = {
    "feature_col": "raw_waveform",
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "max_len": None,
    "percentile": 90,
    "batch_size": 6,
    "optimizer": "adam",
    "lr": 1e-4,
    "momentum": 0.9,
    "epochs": 100,
    "patience": 10,
    "log_interval": 5,
    "use_audio_type": True,
    "verbose": True,
    "processor_name": "facebook/hubert-base-ls960",
}

In [ ]:
criterion = nn.BCEWithLogitsLoss(reduction='mean')
model, state = train_model(train_df, test_df, model, criterion, **args)

We obtain a validation AUC of 75.09% (95% confidence interval of 70.55% - 79.62%) after 13 epochs. This performance is good, but it isn't an improvement over the `HubertAudioClassifier` baseline. Since there's only two possible audio types, it is likely that either (1) that information can easily be learned and deduced by the model on its own or (2) that information isn't helpful for the classification task at hand.